# Pandas Fundamentals

Pandas is a fast, powerful, flexible, and easy-to-use open-source data analysis and manipulation tool built on top of Python. This notebook covers DataFrame loading, indexing, slicing, masking, groupings, aggregations, and date-time operations.

In [2]:
import pandas as pd

# Load CSV → DataFrame
df = pd.read_csv('studentperformence.csv')

# Shape: (rows, columns)
df.shape       # (1000, 8)

# Preview rows
df.head()      # first 5
df.head(10)   # first 10
df.tail()      # last 5

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77
999,female,group D,some college,free/reduced,none,77,86,86


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


In [ ]:
df.columns    # Index of all column names
df.dtypes     # dtype per column
df.count()    # non-null count per column — race/ethnicity shows 999

In [ ]:
# Single column → Series (1D)
df['gender']
print(type(df['gender']))  # pandas.core.series.Series

# Multiple columns → DataFrame (2D)
df[['gender', 'lunch']]
print(type(df[['gender','lunch']]))  # pandas.core.frame.DataFrame

In [4]:
df.iloc[1:, :]      # rows 1 to end → 999 rows
df.iloc[[0, 3]]     # rows 0 and 3 only
df.iloc[1:3]        # rows 1 and 2 (exclusive end)

# ❌ ERROR — iloc requires numeric indexers
# df.iloc[:, ['gender']]  → IndexError!

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93


In [5]:
df.loc[0]                         # single row
df.loc[[0, 3]]                    # rows 0 and 3
df.loc[0, ['gender']]             # row 0, gender col
df.loc[:, ['gender']]             # all rows, gender only → 1000 rows × 1 col
df.loc[10:, ['gender', 'lunch']] # rows 10+, two cols → 990 rows × 2 cols
df.loc[10:, 'gender':'lunch']    # col slice inclusive → 990 rows × 4 cols

,gender,race/ethnicity,parental level of education,lunch
10,male,group C,associate's degree,standard
11,male,group D,associate's degree,standard
12,female,group B,high school,standard
13,male,group A,some college,standard
14,female,group A,master's degree,standard
...,...,...,...,...
995,female,group E,master's degree,standard
996,male,group C,high school,free/reduced
997,female,group C,high school,free/reduced
998,female,group D,some college,standard


In [ ]:
# Boolean mask
df['gender'] == 'female'   # True/False per row

# Apply mask
df[df['gender'] == 'female']          # 518 rows

# AND: both must be True
df[(df['gender'] == 'female') & (df['math score'] > 70)]  # 170 rows

# AND: male + standard lunch
df[(df['gender'] == 'male') & (df['lunch'] == 'standard')]   # 316 rows

# ❌ WRONG — TypeError without parentheses
# df[df['gender']=='female' & df['math score']>70]

In [ ]:
# unique() — all distinct values
df['gender'].unique()    # array(['female', 'male'])

# nunique() — count of distinct values
df['gender'].nunique()  # 2

# value_counts() — frequency table
df['gender'].value_counts()
# female  518 | male  482

In [ ]:
col_names = ['gender', 'race/ethnicity', 'parental level of education',
             'lunch', 'test preparation course']

# unique values per column
for col in col_names:
    print(f"{col} => {df[col].unique()}")

# nunique per column
for col in col_names:
    print(f"{col} => {df[col].nunique()}")

## axis=0 — Column-wise (default)
## Goes down each column. Returns one value per column — the average across all 1000 students per subject.

In [8]:



df.mean(numeric_only=True, axis=0)

math score       66.089
reading score    69.169
writing score    68.054
dtype: float64

## axis=1 — Row-wise
## Goes across each row. Returns one value per student — their personal average score across all 3 subjects.

In [ ]:
df.mean(numeric_only=True, axis=1)

## 08
# agg() — Multiple Aggregations
# Power Tool

In [ ]:
# Multiple stats on multiple columns
df[['reading score', 'math score', 'writing score']].agg(['mean', 'min', 'max'])

# Dictionary Syntax — Different Function per Column

In [9]:
df.agg({
    'writing score': 'min',
    'reading score': 'mean'
})

writing score    10.000
reading score    69.169
dtype: float64

## 14 groupby() with get_group() and agg()
### Advanced
### 1 Split
### Pandas splits rows into groups by column value (e.g. 'female': 518 rows, 'male': 482 rows).
### 2 Apply
### An aggregation function (mean, sum, count…) is computed independently for each group.
### 3 Combine
### Results are combined back into a single DataFrame indexed by the group values.

In [13]:
# Create GroupBy object
group = df.groupby('gender')



In [14]:
# Inspect group index → shows which rows belong to each group
group.groups   # {'female': [0,1,2,...], 'male': [3,4,...]}


{'female': [0, 1, 2, 5, 6, 9, 12, 14, 15, 17, 19, 21, 23, 27, 29, 30, 31, 32, 36, 37, 38, 41, 42, 44, 46, 47, 48, 54, 55, 56, 59, 63, 64, 67, 69, 70, 72, 78, 79, 80, 85, 86, 87, 88, 89, 90, 94, 97, 98, 99, 102, 105, 106, 108, 109, 110, 113, 114, 116, 117, 118, 119, 120, 122, 125, 129, 133, 138, 140, 141, 142, 145, 148, 152, 155, 156, 158, 161, 164, 165, 167, 168, 169, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 189, 190, 192, 194, 198, 199, ...], 'male': [3, 4, 7, 8, 10, 11, 13, 16, 18, 20, 22, 24, 25, 26, 28, 33, 34, 35, 39, 40, 43, 45, 49, 50, 51, 52, 53, 57, 58, 60, 61, 62, 65, 66, 68, 71, 73, 74, 75, 76, 77, 81, 82, 83, 84, 91, 92, 93, 95, 96, 100, 101, 103, 104, 107, 111, 112, 115, 121, 123, 124, 126, 127, 128, 130, 131, 132, 134, 135, 136, 137, 139, 143, 144, 146, 147, 149, 150, 151, 153, 154, 157, 159, 160, 162, 163, 166, 170, 171, 180, 184, 185, 186, 187, 188, 191, 193, 195, 196, 197, ...]}

In [15]:
# Extract a specific group as a DataFrame
female = group.get_group('female')  # 518 rows
male   = group.get_group('male')    # 482 rows

In [16]:
# Compute mean math score per gender
female['math score'].mean()   # 63.63
male['math score'].mean()     # 68.73


np.float64(68.72821576763485)

In [17]:
# Equivalent using agg on filtered df
female.agg({'math score': 'mean'})

math score    63.633205
dtype: float64

In [18]:
# Most efficient: groupby + agg in one line
group.agg({'math score': 'mean'})

,math score
gender,
female,63.633205
male,68.728216


In [19]:
# ⭐ Core pattern — mean AND min in one call
group.agg({'math score': ['mean', 'min']})

math score    
             mean min
gender               
female  63.633205   0
male    68.728216  27

# Cheet Sheet For pandas PD

In [ ]:
# import pandas as pd

# # ── LOADING ──────────────────────────────────────────────
# df = pd.read_csv("file.csv")

# # ── INSPECTION ───────────────────────────────────────────
# df.shape                     # (rows, cols)
# df.head() / df.tail()         # preview
# df.info()                    # types + null counts
# df.dtypes / df.columns        # types / names
# df.count()                   # non-null per column

# # ── SELECTION ────────────────────────────────────────────
# df['col']                    # Series (1D)
# df[['c1','c2']]             # DataFrame (2D)
# df.iloc[1:,:]               # position-based, exclusive end
# df.loc[10:,'col1':'col2']  # label-based, inclusive end

# # ── FILTERING ────────────────────────────────────────────
# df[df['col'] == 'val']
# df[(df['c1']=='v') & (df['c2']>70)]  # AND
# df[(df['c1']=='v') | (df['c2']>70)]  # OR

# # ── CATEGORICAL ──────────────────────────────────────────
# df['col'].unique()            # distinct values
# df['col'].nunique()           # count distinct (excl NaN)
# df['col'].value_counts()      # frequency table

# # ── AGGREGATION ──────────────────────────────────────────
# df.mean(numeric_only=True, axis=0)  # per column
# df.mean(numeric_only=True, axis=1)  # per row
# df[['c1','c2']].agg(['mean','min','max'])
# df.agg({'c1':'min', 'c2':'mean'})

# # ── MISSING VALUES ───────────────────────────────────────
# df.isnull().sum()             # count per column
# df.isnull().sum().sum()      # total
# df['col'].notna()             # True where not NaN
# df.fillna(0)                  # fill with constant
# df.fillna({'c': df['c'].mean()})  # fill per column
# df.ffill()                    # forward fill
# df.bfill()                    # backward fill

# # ── CLEANING ─────────────────────────────────────────────
# df.dropna()                   # drop rows with any NaN
# df.dropna(how='all')          # drop rows all NaN
# df.dropna(subset=['col'])     # drop if specific col NaN
# df.drop(['c'], axis=1)       # drop column
# df.drop(index=[0,1])          # drop rows
# df.drop_duplicates()           # remove exact duplicate rows

# # ── GROUPBY ──────────────────────────────────────────────
# g = df.groupby('col')
# g.get_group('val')            # extract one group
# g.agg({'score': 'mean'})      # aggregate each group

---
## 🛠️ Working with DateTime in Pandas

*(This section is consolidated from `new.ipynb`)*

Converting objects/strings to true `datetime` types is crucial for time-series analysis and extracting date components (year, month, day, weekday, etc.).

In [7]:
store=pd.read_csv("Superstore.csv")

In [8]:
store.dtypes

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [9]:
store.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [10]:
#store['Order Date']=pd.to_datetime(store['Order Date'],format)
#store['Order Date']=pd.to_datetime(store['Order Date'],format='mixed')

In [11]:
for col in ['Order Date','Ship Date']:
    store[col]=pd.to_datetime(store[col],format='mixed')#

In [12]:
store.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

In [13]:
store.dtypes

Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

In [14]:
store['Order Date']

0      2016-11-08
1      2016-11-08
2      2016-06-12
3      2015-10-11
4      2015-10-11
          ...    
9989   2014-01-21
9990   2017-02-26
9991   2017-02-26
9992   2017-02-26
9993   2017-05-04
Name: Order Date, Length: 9994, dtype: datetime64[ns]

In [19]:
store['year']=store['Order Date'].dt.year
store['month']=store['Order Date'].dt.month
store['day']=store['Order Date'].dt.day


In [22]:
store[['year','month','day']]

,year,month,day
0,2016,11,8
1,2016,11,8
2,2016,6,12
3,2015,10,11
4,2015,10,11
...,...,...,...
9989,2014,1,21
9990,2017,2,26
9991,2017,2,26
9992,2017,2,26


In [33]:
store.head(10)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,year,month,day
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,2016,11,8
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,2016,11,8
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,2016,6,12
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,2015,10,11
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,2015,10,11
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694,2014,2014,6,9
6,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Art,Newell 322,7.2800,4,0.00,1.9656,2014,2014,6,9
7,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152,2014,2014,6,9
8,9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3,0.20,5.7825,2014,2014,6,9
9,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5,0.00,34.4700,2014,2014,6,9


In [35]:
for year in store['year']:
    print(year,store['Profit'].sum())

2016 286397.0217
2016 286397.0217
2016 286397.0217
2015 286397.0217
2015 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2017 286397.0217
2016 286397.0217
2015 286397.0217
2015 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2014 286397.0217
2016 286397.0217
2016 286397.0217
2017 286397.0217
2015 286397.0217
2016 286397.0217
2016 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2017 286397.0217
2016 286397.0217
2016 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2017 286397.0217
2016 286397.0217
2017 286397.0217
2016 286397.0217
2016 286397.0217
2014 286397.0217
2016 286397.0217
2016 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2015 286397.0217
2016 286397.0217
2016 286397.0217
2016 286397.0217
2016 286397.0217
2016 286397.0217
2016 286397.02

In [37]:
yearly_profit=store.groupby('year')
yearly_profit['Profit'].sum()

year
2014    49543.9741
2015    61618.6037
2016    81795.1743
2017    93439.2696
Name: Profit, dtype: float64

In [40]:
customer=store.groupby('Customer Name')
customer['Profit'].sum().idxmax()

'Tamara Chand'

In [44]:
store.sort_values('Profit',ascending=False).iloc[0]

Row ID                                            6827
Order ID                                CA-2016-118689
Order Date                         2016-10-02 00:00:00
Ship Date                          2016-10-09 00:00:00
Ship Mode                               Standard Class
Customer ID                                   TC-20980
Customer Name                             Tamara Chand
Segment                                      Corporate
Country                                  United States
City                                         Lafayette
State                                          Indiana
Postal Code                                      47905
Region                                         Central
Product ID                             TEC-CO-10004722
Category                                    Technology
Sub-Category                                   Copiers
Product Name     Canon imageCLASS 2200 Advanced Copier
Sales                                         17499.95
Quantity  

In [50]:
store.tail(5)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,year,month,day
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,Furnishings,Ultra Door Pull Handle,25.248,3,0.2,4.1028,2014,2014,1,21
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.960,2,0.0,15.6332,2017,2017,2,26
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,Phones,Aastra 57i VoIP phone,258.576,2,0.2,19.3932,2017,2017,2,26
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.600,4,0.0,13.3200,2017,2017,2,26
9993,9994,CA-2017-119914,2017-05-04,2017-05-09,Second Class,CC-12220,Chris Cortes,Consumer,United States,Westminster,...,Appliances,"Acco 7-Outlet Masterpiece Power Center, Wihtou...",243.160,2,0.0,72.9480,2017,2017,5,4


In [51]:
store.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,year,month,day
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,2016,11,8
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,2016,11,8
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,2016,6,12
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,2015,10,11
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,2015,10,11


In [58]:
store['Customer Name'].str.split()[0]

['Claire', 'Gute']

In [59]:
store['Customer Name'].str.split().str[0]

0       Claire
1       Claire
2       Darrin
3         Sean
4         Sean
         ...  
9989       Tom
9990      Dave
9991      Dave
9992      Dave
9993     Chris
Name: Customer Name, Length: 9994, dtype: object

In [68]:
store['Customer Name'].str.split().str[-1]

0               Gute
1               Gute
2               Huff
3          O'Donnell
4          O'Donnell
            ...     
9989    Boeckenhauer
9990          Brooks
9991          Brooks
9992          Brooks
9993          Cortes
Name: Customer Name, Length: 9994, dtype: object

In [66]:
store.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,year,month,day
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,2016,11,8
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,2016,11,8
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,2016,6,12
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,2015,10,11
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,2015,10,11


In [76]:
store['Sales']=store["Sales"].astype(int)

In [77]:
store.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   datetime64[ns]
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

np.float64(-6599.978)